In [199]:
import ROOT
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import os

In [200]:

# Function to get histogram from ROOT file
def get_histogram(file_path, name, cat):
    file = ROOT.TFile.Open(f"{file_path}/{name}.root")
    hist = file.Get(f"{name}_{cat}")
    hist.SetDirectory(0)  # Detach the histogram from the file
    file.Close()
    return hist

# Convert ROOT histograms to numpy arrays for plotting
def hist_to_array(hist):
    bins = hist.GetNbinsX()
    bin_edges = np.array([hist.GetBinLowEdge(i+1) for i in range(bins+1)])
    bin_contents = np.array([hist.GetBinContent(i+1) for i in range(bins)])
    return bin_edges, bin_contents

def add_legends(ax, legends, loc='upper right', spacing=0.26):
    """
    Add multiple legends to a Matplotlib axes.

    Parameters:
        ax: The axes to which legends are added.
        legends: A list of legend handles, each a list of Line2D objects to be included in a legend.
        loc: Location of the legends (default 'upper right').
        spacing: Horizontal space between legends (default 0.1).
    """
    num_legends = len(legends)
    base_anchor = (0.995, 0.995)
    
    for i, handles in enumerate(legends):
        # Adjust the x-coordinate of the bbox_to_anchor for each legend
        x_offset = base_anchor[0] - i * spacing
        legend = ax.legend( handles=handles, loc=loc, bbox_to_anchor=(x_offset, base_anchor[1]))
        ax.add_artist(legend)
        legend.get_frame().set_linewidth(0.0)



## Plot multiple mediator masses

In [50]:

cat = "0SVJ"
wp = "1.00"
# Define the file path and output names
file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/wp-{wp}/" + cat
# file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/alldata-wp-{wp}/" + cat


# Background names
# background_names = ["QCD", "TTbar", "Wjets", "Zjets"]
background_names = ["QCD", "TTbar", "Zjets"]


# Define the signal you want to plot
signal_name = [
    "SVJ_700_0p3", "SVJ_700_0p5", "SVJ_700_0p7",
    # "SVJ_800_0p3", "SVJ_800_0p5", "SVJ_800_0p7",
    # "SVJ_900_0p3", "SVJ_900_0p5", "SVJ_900_0p7",
    "SVJ_1000_0p3", "SVJ_1000_0p5", "SVJ_1000_0p7",
    # "SVJ_1100_0p3", "SVJ_1100_0p5", "SVJ_1100_0p7",
    # "SVJ_1200_0p3", "SVJ_1200_0p5", "SVJ_1200_0p7",
    # "SVJ_1300_0p3", "SVJ_1300_0p5", "SVJ_1300_0p7",
    # "SVJ_1400_0p3", "SVJ_1400_0p5", "SVJ_1400_0p7",
    "SVJ_1500_0p3", "SVJ_1500_0p5", "SVJ_1500_0p7",
    # "SVJ_3000_0p3", "SVJ_3000_0p5", "SVJ_3000_0p7",
]
labels = ["$M_{Z'} = 700$ GeV", "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1500$ GeV",]
colors = ['plum','#E63946','#1D3557']
output_filename = "MultipleMz_rinv05_part1.png"


# signal_name = [
#     # "SVJ_700_0p3", "SVJ_700_0p5", "SVJ_700_0p7",
#     "SVJ_800_0p3", "SVJ_800_0p5", "SVJ_800_0p7",
#     # "SVJ_900_0p3", "SVJ_900_0p5", "SVJ_900_0p7",
#     "SVJ_1000_0p3", "SVJ_1000_0p5", "SVJ_1000_0p7",
#     # "SVJ_1100_0p3", "SVJ_1100_0p5", "SVJ_1100_0p7",
#     "SVJ_1200_0p3", "SVJ_1200_0p5", "SVJ_1200_0p7",
#     # "SVJ_1300_0p3", "SVJ_1300_0p5", "SVJ_1300_0p7",
#     "SVJ_1400_0p3", "SVJ_1400_0p5", "SVJ_1400_0p7",
#     # "SVJ_1500_0p3", "SVJ_1500_0p5", "SVJ_1500_0p7",
#     # "SVJ_3000_0p3", "SVJ_3000_0p5", "SVJ_3000_0p7",
# ]
# colors = ['palevioletred', 'gold', 'olive', 'lightseagreen', '#14243A']
# labels = ["$M_{Z'} = 800$ GeV", "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1200$ GeV","$M_{Z'} = 1400$ GeV"]
# output_filename = "MultipleMz_rinv05_part2.png"


# Initialize an empty list to store background histograms
background_histograms = {}
total_bkg = []

In [51]:
# Retrieve and sum background histograms
for name in background_names:
    hist = get_histogram(file_path, name, cat)
    if not total_bkg:
        total_bkg.append(hist.Clone())
    else:
        total_bkg[0].Add(hist)


# Get data for plotting
tot_bin_edges, tot_bin_contents = hist_to_array(total_bkg[0])

In [52]:
# Retrieve and sum background histograms
for name in background_names:
    hist = get_histogram(file_path, name, cat)
    background_histograms[name] = hist


signal_hist = {}
# Retrieve the signal histogram
for name in signal_name:
    hist = get_histogram(file_path, name, cat)
    signal_hist[name] = hist




# Get data for plotting
bg_bin_edges= {}
bg_bin_contents = {}
for hist in background_histograms:
    bg_bin_edges[hist],bg_bin_contents[hist] = hist_to_array(background_histograms[hist])

sig_bin_edges = {}
sig_bin_contents = {}
for hist in signal_hist:
    sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])


In [55]:

xmin, xmax, nbins, label = 650,2000,25,"$M_{T}$"

rinv = ["r03", "r05", "r07"]

linestyles = ['-', '--', ':','-', '--', ':','-', '--', ':']
line_style_with_label = {"$r_{inv} = 0.3$": '-', "$r_{inv} = 0.5$": '--', '$r_{inv} = 0.7$': ':'}
color_bkg = ["darkgrey", "silver", "gainsboro"]
fig, ax = plt.subplots(figsize=(8, 8), dpi = 200)
ax.set_yscale('log')

colors = ['plum','plum','plum','#E63946','#E63946','#E63946','#1D3557','#1D3557','#1D3557']



# Background
for i, hist in enumerate(background_histograms):
    ax.hist(bg_bin_edges [hist][:-1], bins=nbins, weights = bg_bin_contents[hist], range=(xmin, xmax),  histtype='stepfilled', density = False, stacked=True, color=color_bkg[i], linewidth=1)


for i, hist in enumerate(signal_hist):
    # # ax.hist(tot_bin_edges[:-1], bins=nbins, weights = tot_bin_contents, range=(xmin, xmax), density=False, stacked =True, histtype='stepfilled', color='gainsboro', linewidth=1, alpha = 1)
    ax.hist(sig_bin_edges[hist][:-1], bins=nbins, weights= sig_bin_contents[hist], range=(xmin, xmax),  histtype='step', density= False, stacked=False, color=colors[i], linestyle=linestyles[i], linewidth=1)
 

ax.set_xlabel(label)
ax.xaxis.set_label_coords(0.95, -0.05)
ax.set_ylabel("a.u.")
ax.yaxis.set_label_coords(-0.1, 0.95)
y_min, y_max = ax.get_ylim()
ax.set_ylim(y_min, y_max + 0.01)



legend_handles_1 = [plt.Line2D([0], [1], color='black', linewidth = 0.9,linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
legend_handles_2 = [plt.Line2D([0], [1], color=color, linewidth=0.9, linestyle='-', label=Mz) for Mz, color in zip(labels, colors)]
additional_handle_2 = [plt.Line2D([0], [1], color=color, linewidth=0.9, linestyle='',marker="s", label=bkg) for bkg , color in zip(background_names, color_bkg)]
# legend_handles_1 = legend_handles_1 + [additional_handle_2]
legend_handles_2 = legend_handles_2  +  additional_handle_2


add_legends(ax, [legend_handles_2,legend_handles_1])

output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"

if not os.path.exists(output_path):
    os.mkdir(output_path)
    print(f"Created directory: {output_path}")

output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/" + cat +"/"

if not os.path.exists(output_path):
    os.mkdir(output_path)
    print(f"Created directory: {output_path}")

plt.savefig(output_path + output_filename)
print(f"Plot save to {output_path + output_filename}")
plt.close(fig)
print(" ==> Plot saved")









Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-1.00/0SVJ/MultipleMz_rinv05_part1.png
 ==> Plot saved


### automate

In [4]:

list_cat = ["0SVJ","1SVJ", "2SVJ"]
list_rinv = ["rinv03", "rinv05", "rinv07"]
wp = "0.50"


# Background names
background_names = ["QCD", "TTbar", "Wjets", "Zjets"]

# Define the signal you want to plot
signal_names = [[
    "SVJ_700_0p3", "SVJ_700_0p5", "SVJ_700_0p7",
    "SVJ_900_0p3", "SVJ_900_0p5", "SVJ_900_0p7",
    "SVJ_1100_0p3", "SVJ_1100_0p5", "SVJ_1100_0p7",
    "SVJ_1300_0p3", "SVJ_1300_0p5", "SVJ_1300_0p7",
    "SVJ_1500_0p3", "SVJ_1500_0p5", "SVJ_1500_0p7",],

    ["SVJ_800_0p3", "SVJ_800_0p5", "SVJ_800_0p7",
    "SVJ_1000_0p3", "SVJ_1000_0p5", "SVJ_1000_0p7",
    "SVJ_1200_0p3", "SVJ_1200_0p5", "SVJ_1200_0p7",
    "SVJ_1400_0p3", "SVJ_1400_0p5", "SVJ_1400_0p7",
    ]]
list_labels = [["$M_{Z'} = 700$ GeV", "$M_{Z'} = 900$ GeV","$M_{Z'} = 1100$ GeV","$M_{Z'} = 1300$ GeV","$M_{Z'} = 1500$ GeV",],
    ["$M_{Z'} = 800$ GeV", "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1200$ GeV","$M_{Z'} = 1400$ GeV"]]
list_colors = [['plum','#E63946','#F4A261','#A8DADC','#1D3557'],
    ['palevioletred', 'gold', 'olive', 'lightseagreen', '#14243A']]




for cat in list_cat:
    file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-{wp}/" + cat
    # file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/alldata-wp-{wp}/" + cat

    for rinv in list_rinv:
        output_filenames = [f"MultipleMz_{rinv}_part1.png", f"MultipleMz_{rinv}_part2.png"]

        for signal_name, labels, colors, output_filename in zip(signal_names, list_labels, list_colors, output_filenames):

            # Initialize an empty list to store background histograms
            background_histograms = []

            xmin, xmax, nbins, label = 650,2000,25,"$M_{T}$"

            rinv = ["r03", "r05", "r07"]

            linestyles = ['-', '-', '-']
            line_style_with_label = {"$r_{inv} = 0.3$": '-', "$r_{inv} = 0.5$": '--', '$r_{inv} = 0.7$': ':'}

            fig, ax = plt.subplots(figsize=(8, 8),dpi = 200)
            ax.set_yscale('log')
            
            # Retrieve and sum background histograms
            for name in background_names:
                hist = get_histogram(file_path, name, cat)
                if not background_histograms:
                    background_histograms.append(hist.Clone())
                else:
                    background_histograms[0].Add(hist)

            # Get data for plotting
            bg_bin_edges, bg_bin_contents = hist_to_array(background_histograms[0])

            # Background
            nbkg, binsbkg, patchesbkg = ax.hist(bg_bin_edges[:-1], bins=nbins, weights = bg_bin_contents, range=(xmin, xmax), density=True, histtype='stepfilled', color='gainsboro', linewidth=1)

            # for patch in patchesbkg:
            #     patch.set_hatch('////')
            #     patch.set_edgecolor("gainsboro")
            #     patch.set_linewidth(1.2) 
            
            signal_hist = {}
            # Retrieve the signal histogram
            for name in signal_name:
                hist = get_histogram(file_path, name, cat)
                signal_hist[name] = hist
                
            sig_bin_edges = {}
            sig_bin_contents = {}
            for hist in signal_hist:
                sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])

            i_color = 0
            for i, hist in enumerate(signal_hist):
                i_line =  i % 3 
                if i % 3 == 1:
                    ax.hist(sig_bin_edges[hist][:-1], bins=nbins, weights= sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=colors[i_color], linestyle=linestyles[i_line], linewidth=1)
                if i_line == 2:
                    i_color += 1

            ax.set_xlabel(label)
            ax.xaxis.set_label_coords(0.95, -0.05)
            ax.set_ylabel("a.u.")
            ax.yaxis.set_label_coords(-0.1, 0.95)
            y_min, y_max = ax.get_ylim()
            ax.set_ylim(y_min, y_max + 0.01)



            # legend_handles_1 = [plt.Line2D([0], [1], color="black", linewidth=0.9, linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
            legend_handles_2 = [plt.Line2D([0], [1], color=color, linewidth=0.9, linestyle='-', label=Mz) for Mz, color in zip(labels, colors)]
            additional_handle_2 = plt.Line2D([0], [1], color='gainsboro', linewidth=0.9, linestyle='-', label='QCD, TTbar, EW')
            # legend_handles_1 = legend_handles_1 + [additional_handle_2]
            legend_handles_2 = legend_handles_2  + [additional_handle_2]


            add_legends(ax, [legend_handles_2,])

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/" + cat +"/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            plt.savefig(output_path + output_filename)
            print(f"Plot save to {output_path + output_filename}")
            plt.close(fig)
            print(" ==> Plot saved")









Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv03_part1.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv03_part2.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv05_part1.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv05_part2.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv07_part1.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv07_part2.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleMz_rinv03_part1.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleMz_rinv03_part2.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleM

In [204]:
list_cat = ["0SVJ","1SVJ", "2SVJ"]
list_rinv = ["rinv03", "rinv05", "rinv07"]
wp = "0.50_finebinning"


# Background names
background_names = ["QCD", "TTbar", "Wjets", "Zjets"]

# Define the signal you want to plot
signal_names = [[
    "SVJ_900_0p3", "SVJ_900_0p5", "SVJ_900_0p7",
    "SVJ_1000_0p3", "SVJ_1000_0p5", "SVJ_1000_0p7",
    "SVJ_1100_0p3", "SVJ_1100_0p5", "SVJ_1100_0p7",],
     ]
list_labels = [["$M_{Z'} = 900$ GeV", "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1100$ GeV"],]
list_colors = [['plum','#E63946','#1D3557']]
# list_labels = [[ "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1500$ GeV"],]
# list_colors = [['#E63946','#1D3557']]



for cat in list_cat:
    file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-{wp}/" + cat
    # file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/alldata-wp-{wp}/" + cat

    for k,rinv in enumerate(list_rinv):
        output_filenames = [f"MultipleMz_{rinv}.png"]

        for signal_name, labels, colors, output_filename in zip(signal_names, list_labels, list_colors, output_filenames):

            # Initialize an empty list to store background histograms
            background_histograms = []

            xmin, xmax, nbins, label = 650,2000,20,"$M_{T}$"
            # nbins = [650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200,1300, 1400,1600, 2000]

            rinv = ["r03", "r05", "r07"]

            linestyles = ['-', '-', '-']
            line_style_with_label = {"$r_{inv} = 0.3$": '-', "$r_{inv} = 0.5$": '--', '$r_{inv} = 0.7$': ':'}

            fig, ax = plt.subplots(figsize=(8, 8),dpi = 200)
            ax.set_yscale('log')
            
            # Retrieve and sum background histograms
            for name in background_names:
                hist = get_histogram(file_path, name, cat)
                if not background_histograms:
                    background_histograms.append(hist.Clone())
                else:
                    background_histograms[0].Add(hist)

            # Get data for plotting
            bg_bin_edges, bg_bin_contents = hist_to_array(background_histograms[0])

            # Background
            nbkg, binsbkg, patchesbkg = ax.hist(bg_bin_edges[:-1], bins=nbins, weights = bg_bin_contents, range=(xmin, xmax), density=True, histtype='stepfilled', color='gainsboro', linewidth=1)

            # for patch in patchesbkg:
            #     patch.set_hatch('////')
            #     patch.set_edgecolor("gainsboro")
            #     patch.set_linewidth(1.2) 
            
            signal_hist = {}
            # Retrieve the signal histogram
            for name in signal_name:
                hist = get_histogram(file_path, name, cat)
                signal_hist[name] = hist
                
            sig_bin_edges = {}
            sig_bin_contents = {}
            for hist in signal_hist:
                sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])

            i_color = 0
            for i, hist in enumerate(signal_hist):
                if hist in ["SVJ_700_0p3", "SVJ_700_0p5", "SVJ_700_0p7",]:
                    xmin, xmax, nbins, label = 650,2000,15,"$M_{T}$"
                i_line =  i % 3 
                if i % 3 == k:
                    ax.hist(sig_bin_edges[hist][:-1], bins=nbins, weights= sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=colors[i_color], linestyle=linestyles[i_line], linewidth=1.5)
                if i_line == k:
                    i_color += 1

            ax.set_xlabel(label)
            ax.xaxis.set_label_coords(0.95, -0.05)
            ax.set_ylabel("a.u.")
            ax.yaxis.set_label_coords(-0.1, 0.95)
            y_min, y_max = ax.get_ylim()
            ax.set_ylim(y_min, y_max + 0.01)



            # legend_handles_1 = [plt.Line2D([0], [1], color="black", linewidth=0.9, linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
            legend_handles_2 = [plt.Line2D([0], [1], color=color, linewidth=0.9, linestyle='-', label=Mz) for Mz, color in zip(labels, colors)]
            additional_handle_2 = plt.Line2D([0], [1], color='gainsboro', linewidth=0.9, linestyle='-', label='QCD, TTbar, EW')
            # legend_handles_1 = legend_handles_1 + [additional_handle_2]
            legend_handles_2 = legend_handles_2  + [additional_handle_2]


            add_legends(ax, legends=[legend_handles_2])

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/" + cat +"_new2/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            plt.savefig(output_path + output_filename)
            print(f"Plot save to {output_path + output_filename}")
            plt.close(fig)
            print(" ==> Plot saved")









Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/0SVJ_new2/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/0SVJ_new2/MultipleMz_rinv05.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/0SVJ_new2/MultipleMz_rinv07.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/1SVJ_new2/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/1SVJ_new2/MultipleMz_rinv05.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/1SVJ_new2/MultipleMz_rinv07.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/2SVJ_new2/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.50_finebinning/2SVJ_new2/MultipleMz_rinv05.png
 ==> Plo

In [10]:

list_cat = ["0SVJ","1SVJ", "2SVJ"]
list_rinv = ["rinv03", "rinv05", "rinv07"]
wp = "0.60"


# Background names
background_names = ["QCD", "TTbar", "Wjets", "Zjets"]

# Define the signal you want to plot
signal_names = [[
    "SVJ_700_0p3", "SVJ_700_0p5", "SVJ_700_0p7",
    # "SVJ_900_0p3", "SVJ_900_0p5", "SVJ_900_0p7",
    "SVJ_1000_0p3", "SVJ_1100_0p5", "SVJ_1100_0p7",
    # "SVJ_1300_0p3", "SVJ_1300_0p5", "SVJ_1300_0p7",
    "SVJ_1500_0p3", "SVJ_1500_0p5", "SVJ_1500_0p7",],

    # ["SVJ_800_0p3", "SVJ_800_0p5", "SVJ_800_0p7",
    # "SVJ_1000_0p3", "SVJ_1000_0p5", "SVJ_1000_0p7",
    # "SVJ_1200_0p3", "SVJ_1200_0p5", "SVJ_1200_0p7",
    # "SVJ_1400_0p3", "SVJ_1400_0p5", "SVJ_1400_0p7",
     ]
list_labels = [["$M_{Z'} = 700$ GeV", "$M_{Z'} = 100$ GeV","$M_{Z'} = 1500$ GeV"],]
    # ["$M_{Z'} = 800$ GeV", "$M_{Z'} = 1000$ GeV","$M_{Z'} = 1200$ GeV","$M_{Z'} = 1400$ GeV"]]
list_colors = [['plum','#E63946','#1D3557'],
    ['palevioletred', 'gold', 'olive', 'lightseagreen', '#14243A']]




for cat in list_cat:
    file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-{wp}/" + cat
    # file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/alldata-wp-{wp}/" + cat

    for rinv in list_rinv:
        output_filenames = [f"MultipleMz_{rinv}.png"]

        for signal_name, labels, colors, output_filename in zip(signal_names, list_labels, list_colors, output_filenames):

            # Initialize an empty list to store background histograms
            background_histograms = []

            xmin, xmax, nbins, label = 650,2000,25,"$M_{T}$"

            rinv = ["r03", "r05", "r07"]

            linestyles = ['-', '-', '-']
            line_style_with_label = {"$r_{inv} = 0.3$": '-', "$r_{inv} = 0.5$": '--', '$r_{inv} = 0.7$': ':'}

            fig, ax = plt.subplots(figsize=(8, 8),dpi = 200)
            ax.set_yscale('log')
            
            # Retrieve and sum background histograms
            for name in background_names:
                hist = get_histogram(file_path, name, cat)
                if not background_histograms:
                    background_histograms.append(hist.Clone())
                else:
                    background_histograms[0].Add(hist)

            # Get data for plotting
            bg_bin_edges, bg_bin_contents = hist_to_array(background_histograms[0])

            # Background
            nbkg, binsbkg, patchesbkg = ax.hist(bg_bin_edges[:-1], bins=nbins, weights = bg_bin_contents, range=(xmin, xmax), density=True, histtype='stepfilled', color='gainsboro', linewidth=1)

            # for patch in patchesbkg:
            #     patch.set_hatch('////')
            #     patch.set_edgecolor("gainsboro")
            #     patch.set_linewidth(1.2) 
            
            signal_hist = {}
            # Retrieve the signal histogram
            for name in signal_name:
                hist = get_histogram(file_path, name, cat)
                signal_hist[name] = hist
                
            sig_bin_edges = {}
            sig_bin_contents = {}
            for hist in signal_hist:
                sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])

            i_color = 0
            for i, hist in enumerate(signal_hist):
                i_line =  i % 3 
                if i % 3 == 1:
                    ax.hist(sig_bin_edges[hist][:-1], bins=nbins, weights= sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=colors[i_color], linestyle=linestyles[i_line], linewidth=1)
                if i_line == 2:
                    i_color += 1

            ax.set_xlabel(label)
            ax.xaxis.set_label_coords(0.95, -0.05)
            ax.set_ylabel("a.u.")
            ax.yaxis.set_label_coords(-0.1, 0.95)
            y_min, y_max = ax.get_ylim()
            ax.set_ylim(y_min, y_max + 0.01)



            # legend_handles_1 = [plt.Line2D([0], [1], color="black", linewidth=0.9, linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
            legend_handles_2 = [plt.Line2D([0], [1], color=color, linewidth=0.9, linestyle='-', label=Mz) for Mz, color in zip(labels, colors)]
            additional_handle_2 = plt.Line2D([0], [1], color='gainsboro', linewidth=0.9, linestyle='-', label='QCD, TTbar, EW')
            # legend_handles_1 = legend_handles_1 + [additional_handle_2]
            legend_handles_2 = legend_handles_2  + [additional_handle_2]


            add_legends(ax, [legend_handles_2,])

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/" + cat +"/"

            if not os.path.exists(output_path):
                os.mkdir(output_path)
                print(f"Created directory: {output_path}")

            plt.savefig(output_path + output_filename)
            print(f"Plot save to {output_path + output_filename}")
            plt.close(fig)
            print(" ==> Plot saved")









Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv05.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/0SVJ/MultipleMz_rinv07.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleMz_rinv05.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/1SVJ/MultipleMz_rinv07.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/2SVJ/MultipleMz_rinv03.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/2SVJ/MultipleMz_rinv05.png
 ==> Plot saved
Plot save to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-0.60/2SVJ/MultipleMz_rinv07.png
 ==> Plot saved


## Plot only one mediator mass

In [7]:
def add_legends(ax, title, legends, loc='upper right', spacing=0.26):
    """
    Add multiple legends to a Matplotlib axes.

    Parameters:
        ax: The axes to which <legends are added.
        legends: A list of legend handles, each a list of Line2D objects to be included in a legend.
        loc: Location of the legends (default 'upper right').
        spacing: Horizontal space between legends (default 0.1).
    """
    num_legends = len(legends)
    base_anchor = (0.995, 0.995)
    
    for i, handles in enumerate(legends):
        # Adjust the x-coordinate of the bbox_to_anchor for each legend
        x_offset = base_anchor[0] - i * spacing
        legend = ax.legend(title = title, handles=handles, loc=loc, bbox_to_anchor=(x_offset, base_anchor[1]))
        ax.add_artist(legend)
        legend.get_frame().set_linewidth(0.0)

In [192]:
cat = "0SVJ"
wp = "1.00"
# Define the file path and output names
# file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-{wp}/" + cat
file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/alldata-wp-{wp}/" + cat

file_path2 =f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-0.50_finebinning/2SVJ/"


# Background names
background_names = ["QCD", "TTbar", "Wjets", "Zjets"]

# Define the signal you want to plot
signal_name = [
    # "SVJ_700_0p3",
    #  "SVJ_700_0p5",
    #  "SVJ_700_0p7",
    # "SVJ_800_0p3", 
    # "SVJ_800_0p5",
    #  "SVJ_800_0p7",
    # "SVJ_900_0p3", 
    # "SVJ_900_0p5",
    #  "SVJ_900_0p7",
    # "SVJ_1000_0p3",
    #  "SVJ_1000_0p5",
    #  "SVJ_1000_0p7",
    # "SVJ_1100_0p3",
    #  "SVJ_1100_0p5",
    #  "SVJ_1100_0p7",
    # "SVJ_1200_0p3", 
    # "SVJ_1200_0p5", 
    # "SVJ_1200_0p7",
    # "SVJ_1300_0p3", 
    # "SVJ_1300_0p5", 
    # "SVJ_1300_0p7",
    # "SVJ_1400_0p3", 
    # "SVJ_1400_0p5", 
    # "SVJ_1400_0p7",
  #  "SVJ_1500_0p3",
    #  "SVJ_1500_0p5", 
     "SVJ_1500_0p7",
    # "SVJ_3000_0p3", 
    # "SVJ_3000_0p5", 
    # "SVJ_3000_0p7",
]


# Initialize an empty list to store background histograms
background_histograms = []


In [193]:


# Retrieve and sum background histograms
for name in background_names:
    hist = get_histogram(file_path, name, cat)
    if not background_histograms:
        background_histograms.append(hist.Clone())
    else:
        background_histograms[0].Add(hist)
    
    

signal_hist = {}
# Retrieve the signal histogram
for name in signal_name:
    hist = get_histogram(file_path, name, cat)
    signal_hist[name] = hist

for name in signal_name:
    hist = get_histogram(file_path2, name, "2SVJ")
    signal_hist[name+"_2SVJ"] = hist
    

# Get data for plotting
bg_bin_edges, bg_bin_contents = hist_to_array(background_histograms[0])

background_histograms = []
for name in background_names:
    hist = get_histogram(file_path2, name, "2SVJ")
    if not background_histograms:
        background_histograms.append(hist.Clone())
    else:
        background_histograms[0].Add(hist)
        

bg_bin_edges_2, bg_bin_contents_2 = hist_to_array(background_histograms[0])



sig_bin_edges = {}
sig_bin_contents = {}
for hist in signal_hist:
    sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])


In [194]:
xmin, xmax, nbins, label = 650,2000,20,"$M_{T}$"

rinv = ["r03", "r05", "r07"]
color = "firebrick"
linestyles = ['--', '-']
line_style_with_label = {"Signal Event Selection": '--', "Signal 2SVJ": '-', }
title_name = signal_name[0].split("_")[1]
title = f"$M_{{Z'}} = {title_name}$ GeV"

# Define bins consistently for both background and signal
bins = np.linspace(xmin, xmax, nbins + 1)  # Define the bins

# Calculate the bin centers for the ratio plot
bin_centers = 0.5 * (bins[:-1] + bins[1:])  # Midpoints of each bin

# Setup figure with gridspec
fig = plt.figure(figsize=(8, 8))
gs = fig.add_gridspec(2, height_ratios=[3, 1], hspace=0.05)  # Main plot and ratio plot

# Main plot (top)
ax_main = fig.add_subplot(gs[0])
ax_main.set_yscale('log')

# Background histograms with consistent binning
nbkg, _, _ = ax_main.hist(bg_bin_edges[:-1], bins=bins, weights=bg_bin_contents, range=(xmin, xmax), density=True, histtype='step', color='#457B9D', linewidth=1.5, linestyle="--", label="Event Selection")
nbkg_2, _, _ = ax_main.hist(bg_bin_edges_2[:-1], bins=bins, weights=bg_bin_contents_2, range=(xmin, xmax), density=True, histtype='step', color='#457B9D', linewidth=1.5, linestyle="-", label="2SVJ")

# Signal histograms with consistent binning
# Signal histograms
for i, hist in enumerate(signal_hist):
    if i == 0:
        signal_sel, bins_sel, _= ax_main.hist(sig_bin_edges[hist][:-1], bins=nbins, weights=sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=color, linestyle=linestyles[i], linewidth=1.5)
    if i == 1:
        signal_2SVJ, bins_2SVJ, _= ax_main.hist(sig_bin_edges[hist][:-1], bins=nbins, weights=sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=color, linestyle=linestyles[i], linewidth=1.5)

# Main plot labels
ax_main.set_xlabel(label)
ax_main.xaxis.set_label_coords(0.95, -0.05)
ax_main.set_ylabel("a.u.")
ax_main.yaxis.set_label_coords(-0.1, 0.95)
y_min, y_max = ax_main.get_ylim()
ax_main.set_ylim(y_min, y_max + 0.01)

# Create custom legend
legend_handles_1 = [plt.Line2D([0], [1], color=color, linewidth=1.5, linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
additional_handle_2 = plt.Line2D([0], [1], color='#457B9D', linewidth=1.5, linestyle='--', label='Background Event Selection')
additional_handle_3 = plt.Line2D([0], [1], color='#457B9D', linewidth=1.5, linestyle='-', label='Background 2 SVJ')
legend_handles_1 += [additional_handle_2, additional_handle_3]

add_legends(ax_main, title, [legend_handles_1])

# Ratio plot (bottom)
ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)

# Avoid zero division by setting a threshold
eps = 1e-10

# Calculate background ratio
bg_ratio = np.divide(nbkg, nbkg_2, out=np.zeros_like(nbkg), where=(nbkg_2 > eps))

# Plot background ratio using bin centers
ax_ratio.step(bin_centers, bg_ratio, where='mid', color='#457B9D', linestyle='-', linewidth=1.5, label='Background Ratio')

# Calculate signal ratio between the two signal histograms
signal_ratio = np.divide(signal_sel, signal_2SVJ, out=np.zeros_like(signal_sel), where=(signal_2SVJ > eps))

# Plot signal ratio using bin centers
ax_ratio.step(bin_centers, signal_ratio, where='mid', color=color, linestyle='-', linewidth=1.5, label='Signal Ratio')

# Formatting the ratio plot
ax_ratio.set_ylabel("Ratio")
ax_ratio.legend(loc='upper left')

# Remove x labels from main plot and set x label for ratio plot
plt.setp(ax_main.get_xticklabels(), visible=False)
ax_ratio.set_xlabel(label)

# Save the plot
output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Created directory: {output_path}")

output_path = f"{output_path}{cat}/"
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Created directory: {output_path}")

plt.savefig(output_path + f"{signal_name[0]}.png")
print(f"Plot saved to {output_path + signal_name[0]}.png")
plt.close(fig)


Plot saved to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-1.00/0SVJ/SVJ_1500_0p7.png


For presentation

In [195]:
cat = "2SVJ"
wp = "1.00"
# Define the file path and output names
# file_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-{wp}/" + cat

file_path =f"/t3home/rhampp/SVJ_GNN/MtDistributions/LundNet5+_SVJ-QCD-TTJet-EW_vs12/wp-0.50_finebinning/2SVJ/"


# Background names
background_names = ["QCD", "TTbar", "Wjets", "Zjets"]

# Define the signal you want to plot
signal_name = [
    # "SVJ_700_0p3",
    #  "SVJ_700_0p5",
    #  "SVJ_700_0p7",
    # "SVJ_800_0p3", 
    # "SVJ_800_0p5",
    #  "SVJ_800_0p7",
    "SVJ_900_0p3", 
    # "SVJ_900_0p5",
    #  "SVJ_900_0p7",
    "SVJ_1000_0p3",
    #  "SVJ_1000_0p5",
    #  "SVJ_1000_0p7",
    "SVJ_1100_0p3",
    #  "SVJ_1100_0p5",
    #  "SVJ_1100_0p7",
    # "SVJ_1200_0p3", 
    # "SVJ_1200_0p5", 
    # "SVJ_1200_0p7",
    # "SVJ_1300_0p3", 
    # "SVJ_1300_0p5", 
    # "SVJ_1300_0p7",
    # "SVJ_1400_0p3", 
    # "SVJ_1400_0p5", 
    # "SVJ_1400_0p7",
  #  "SVJ_1500_0p3",
    #  "SVJ_1500_0p5", 
    #  "SVJ_1500_0p7",
    # "SVJ_3000_0p3", 
    # "SVJ_3000_0p5", 
    # "SVJ_3000_0p7",
]


# Initialize an empty list to store background histograms
background_histograms = []


In [196]:

# Retrieve and sum background histograms
for name in background_names:
    hist = get_histogram(file_path, name, cat)
    if not background_histograms:
        background_histograms.append(hist.Clone())
    else:
        background_histograms[0].Add(hist)
    
    

signal_hist = {}
# Retrieve the signal histogram
for name in signal_name:
    hist = get_histogram(file_path, name, cat)
    signal_hist[name] = hist

    

# Get data for plotting
bg_bin_edges, bg_bin_contents = hist_to_array(background_histograms[0])



sig_bin_edges = {}
sig_bin_contents = {}
for hist in signal_hist:
    sig_bin_edges[hist], sig_bin_contents[hist] = hist_to_array(signal_hist[hist])


In [197]:
xmin, xmax, nbins, label = 650,2000,20,"$M_{T}$"

rinv = ["r03", "r05", "r07"]
color = "firebrick"
linestyles = ['-', '--', ':']
line_style_with_label = {"$M_{Z'} = 900 GeV $": '-', "$M_{Z'} = 1000 GeV $": '--',"$M_{Z'} = 1100 GeV $": ':', }
title_name = signal_name[0].split("_")[1]
title = f"$M_{{Z'}} = {title_name}$ GeV"

# Define bins consistently for both background and signal
bins = np.linspace(xmin, xmax, nbins + 1)  # Define the bins

# Calculate the bin centers for the ratio plot
bin_centers = 0.5 * (bins[:-1] + bins[1:])  # Midpoints of each bin

# Setup figure with gridspec
fig = plt.figure(figsize=(8, 8))
gs = fig.add_gridspec(2, height_ratios=[3, 1], hspace=0.05)  # Main plot and ratio plot

# Main plot (top)
ax_main = fig.add_subplot(gs[0])
ax_main.set_yscale('log')

# Background histograms with consistent binning
nbkg, _, _ = ax_main.hist(bg_bin_edges[:-1], bins=bins, weights=bg_bin_contents, range=(xmin, xmax), density=True, histtype='step', color='#457B9D', linewidth=1.5, linestyle="--", label="Event Selection")

# Signal histograms with consistent binning
# Signal histograms
for i, hist in enumerate(signal_hist):
    ax_main.hist(sig_bin_edges[hist][:-1], bins=nbins, weights=sig_bin_contents[hist], range=(xmin, xmax), density=True, histtype='step', color=color, linestyle=linestyles[i], linewidth=1.5)

# Main plot labels
ax_main.set_xlabel(label)
ax_main.xaxis.set_label_coords(0.95, -0.05)
ax_main.set_ylabel("a.u.")
ax_main.yaxis.set_label_coords(-0.1, 0.95)
y_min, y_max = ax_main.get_ylim()
ax_main.set_ylim(y_min, y_max + 0.01)

# Create custom legend
legend_handles_1 = [plt.Line2D([0], [1], color=color, linewidth=1.5, linestyle=line, label=label_d) for label_d, line in line_style_with_label.items()]
additional_handle_2 = plt.Line2D([0], [1], color='#457B9D', linewidth=1.5, linestyle='--', label='Background Event Selection')
additional_handle_3 = plt.Line2D([0], [1], color='#457B9D', linewidth=1.5, linestyle='-', label='Background 2 SVJ')
legend_handles_1 += [additional_handle_2, additional_handle_3]

add_legends(ax_main, title, [legend_handles_1])

#
# Save the plot
output_path = f"/t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-{wp}/"
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Created directory: {output_path}")

output_path = f"{output_path}{cat}/"
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Created directory: {output_path}")

plt.savefig(output_path + f"{signal_name[0]}.png")
print(f"Plot saved to {output_path + signal_name[0]}.png")
plt.close(fig)


Created directory: /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-1.00/
Created directory: /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-1.00/2SVJ/
Plot saved to /t3home/rhampp/SVJ_GNN/MtDistributions/plots/wp-1.00/2SVJ/SVJ_900_0p3.png
